In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

# --- FILE PATHS ---
INPUT_PKL = "../data/ukhls/pickles/o_indresp_backfilled.pkl"
NORMALIZED_PKL = "../data/ukhls/pickles/normalized.pkl"
OUTPUT_REPORT = "../data/ukhls/tribe_dna.csv"

# --- CONFIGURATION ---
K_CLUSTERS = 10  # Based on our sanity check

def run_persona_pipeline():
    print(f"--- Phase 1: Loading Pre-Normalized Data ---")
    if not os.path.exists(NORMALIZED_PKL):
        print(f"Error: {NORMALIZED_PKL} not found. Please run normalise_backfilled.ipynb first.")
        return None
        
    df_norm = pd.read_pickle(NORMALIZED_PKL)
    df_orig = pd.read_pickle(INPUT_PKL)

    print(f"--- Phase 2: Clustering (K={K_CLUSTERS}) ---")
    exclude_cols = ['pidp', 'o_sex_dv', 'o_oprlg1']
    cluster_features = [c for c in df_norm.columns if c not in exclude_cols]
    km = KMeans(n_clusters=K_CLUSTERS, init='k-means++', n_init=20, random_state=42)
    df_norm['tribe_id'] = km.fit_predict(df_norm[cluster_features])

    print("--- Phase 3: Generating Human-Readable DNA Table ---")
    df_real = pd.merge(df_orig, df_norm[['pidp', 'tribe_id']], on='pidp')
    
    # CRITICAL FIX FOR REPORTING:
    # Pandas .mean() ignores NaNs. If we don't fill missing pay/commute data with 0 here,
    # the average will only represent the people who *do* work/commute, inflating the number!
    for col in ['o_payn_dv', 'o_fimngrs_dv', 'o_jbttwt', 'o_carmiles']:
        if col in df_real.columns:
            df_real[col] = pd.to_numeric(df_real[col], errors='coerce').fillna(0)
            
    def get_mode(series, default):
        m = series.dropna().mode()
        return m.iloc[0] if not m.empty else default
    
    dna_results = []
    for t_id in range(K_CLUSTERS):
        tribe = df_real[df_real['tribe_id'] == t_id]
        row = {
            'Tribe ID': t_id,
            'Size': len(tribe),
            'Age': tribe['o_age_dv'].mean(),
            'Gender': 'Male' if get_mode(tribe['o_sex_dv'], 2) == 1 else 'Female',
            'English': 'Yes' if get_mode(tribe['o_englang'], 1) == 1 else 'No',
            'Religion': {1:'No Religion', 2:'Christian', 3:'Buddhist', 4:'Hindu', 5:'Jewish', 6:'Muslim', 7:'Sikh', 8:'Other'}.get(get_mode(tribe['o_oprlg1'], 1), 'Other'),
            'Education': {1:'Degree', 2:'Other Higher', 3:'A-Level', 4:'GCSE', 5:'Other', 9:'None'}.get(get_mode(tribe['o_hiqual_dv'], 9), 'Other'),
            'Net Pay': tribe['o_payn_dv'].mean(),
            'Gross Pay': tribe['o_fimngrs_dv'].mean(),
            'HH Size': tribe['o_hhsize'].mean(),
            'Children': tribe['o_nchild_dv'].mean(),
            'Settlement': 'Urban' if get_mode(tribe['o_urban_dv'], 2) == 1 else 'Rural',
            'Mental Health': tribe['o_sf12mcs_dv'].mean(),
            'Physical Health': tribe['o_sf12pcs_dv'].mean(),
            'Commute (m)': tribe['o_jbttwt'].mean(),
            'Car Miles': tribe['o_carmiles'].mean(),
            'Shopping(1-5)': tribe['o_locserd'].mean(),
            'Transport(1-5)': tribe['o_locserc'].mean(),
            'Leisure(1-5)': tribe['o_locsere'].mean(),
            'Cohesion': tribe['o_nbrsnci_dv'].mean()
        }
        dna_results.append(row)

    dna_table = pd.DataFrame(dna_results).sort_values('Age')
    
    # Remove decimal places by rounding and converting numerics to integers
    for col in dna_table.select_dtypes(include=[np.number]).columns:
        dna_table[col] = dna_table[col].fillna(0).round(0).astype(int)
        
    dna_table.to_csv(OUTPUT_REPORT, index=False)
    
    return dna_table

if __name__ == "__main__":
    final_dna = run_persona_pipeline()
    print("\n--- FINAL PERSONA PROFILES ---")
    print(final_dna.to_string(index=False))

--- Phase 1: Loading Pre-Normalized Data ---
--- Phase 2: Clustering (K=10) ---
--- Phase 3: Generating Human-Readable DNA Table ---

--- FINAL PERSONA PROFILES ---
 Tribe ID  Size  Age Gender English  Religion Education  Net Pay  Gross Pay  HH Size  Children Settlement  Mental Health  Physical Health  Commute (m)  Car Miles  Shopping(1-5)  Transport(1-5)  Leisure(1-5)  Cohesion
        1  6146   30 Female      No     Other      GCSE      310        910        4         0      Urban             44               53            7        634              3               3             3         3
        7  5475   41 Female      No Christian    Degree     1719       2533        3         0      Urban             43               53           34       2662              3               3             3         3
        9  3908   41 Female     Yes     Other    Degree     1009       2218        5         2      Urban             45               52           13       2498              2        